In [27]:
import pandas as pd 
import re
import emoji 
from pathlib import Path 
data_dir = Path('../data')
resume_dataset=pd.read_csv(data_dir/'interim/combined_resume_final.csv')
# resume_dataset2=pd.read_csv(data_dir/'raw/UpdatedResumeDataSet.csv')


In [28]:
#Lecture et affichage Information dataframme
def read_data(data):
    print("\n __Aperçu des données :")
    print(data.head(3))
    # Afficher les informations sur le DataFrame
    print("\n __Informations sur les données :")
    print(data.info())
    print(" \n __Taille : \n",data.shape)
    print("\n __Information sur les Categories: \n",data['Category'].value_counts().reset_index())

#### Lecture et affichage des donnees

In [29]:
resume_dataset.head()

,Category,Resume
0,Database Administrator,"Ability: Installation and Building Server, Run..."
1,Database Administrator,Ability: database management systems administr...
2,Oracle Database Administrator,Ability: Over 4+ years of Experience as Archit...
3,Oracle Database Administrator,"Ability: Oracle Database Administration, Datab..."
4,Oracle Database Administrator,"Ability: Oracle Database Administration, Oracl..."


In [30]:
read_data(resume_dataset)


 __Aperçu des données :
                        Category  \
0         Database Administrator   
1         Database Administrator   
2  Oracle Database Administrator   

                                              Resume  
0  Ability: Installation and Building Server, Run...  
1  Ability: database management systems administr...  
2  Ability: Over 4+ years of Experience as Archit...  

 __Informations sur les données :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12104 entries, 0 to 12103
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  12104 non-null  object
 1   Resume    12104 non-null  object
dtypes: object(2)
memory usage: 189.2+ KB
None
 
 __Taille : 
 (12104, 2)

 __Information sur les Categories: 
                                 Category  count
0                       Security_Analyst    861
1                     Software_Developer    783
2       Web_Developer,Software_Developer    648
3          

### Normalisation et Proprocessing

#### Normalisation Categories

#### __categories separer par -

In [31]:

# Avant traitement
print("Avant normalisation:")
print(resume_dataset['Category'].nunique(), "catégories uniques")

# Normalisation des catégories
resume_dataset['Category'] = (
    resume_dataset['Category']
    .str.replace('_', ' ')
    .str.title()
    .str.strip()
)
#  Normalisation des variantes "Sr"/"Sr." vers "Senior"
resume_dataset['Category'] = (
    resume_dataset['Category']
    .str.replace(r'\bSr\.?\b', 'Senior', regex=True)  # Capture Sr ET Sr.
    .str.replace(r'Senior\.', 'Senior', regex=True)  # Supprime le point résiduel
    .str.strip()
)
# Après traitement
print("\nAprès normalisation:")
print(resume_dataset['Category'].nunique(), "catégories uniques")

Avant normalisation:
220 catégories uniques

Après normalisation:
182 catégories uniques


#### __Gestion des catégories 


In [32]:
# Identification des catégories rares
category_counts = resume_dataset['Category'].value_counts()
rare_categories = category_counts[category_counts < 30].index
print(f"\nNombre de catégories rares (moins de 20 occurrences): {len(rare_categories)}")
print("\nListe des catégories rares avec leur compte :")
print(rare_categories.sort_values())


Nombre de catégories rares (moins de 20 occurrences): 136

Liste des catégories rares avec leur compte :
Index(['Administrative Assistant', 'Advocate', 'Aem Developer',
       'Android Application Developer', 'Android Developer',
       'Application Developer', 'Arts', 'Automation Testing',
       'Backend Developer', 'Blockchain',
       ...
       'SeniorFullstack Developer', 'Systems Analyst', 'Systems Engineer',
       'Technical Consultant', 'Technical Project Manager', 'Testing',
       'Ui Developer Ui Developer Ui Developer', 'Ui/ Front End Developer',
       'Web Designing', 'Web Developer Web Developer Web Developer'],
      dtype='object', name='Category', length=136)


In [33]:
## Analyse et Groupement categories de faible Occurrence
map_remplacement = {
    # --- Other IT  ---
    "Automation Testing": "Other IT",
    "Blockchain": "Other IT",
    "Director Of It": "Other IT",
    "Help Desk Analyst": "Other IT",
    "Help Desk Technician": "Other IT",
    "It Auditor": "Other IT",
    "Pmo": "Other IT",
    "Program Manager": "Other IT",
    "Technical Consultant": "Other IT",
    "Testing": "Other IT",
    "Desktop Support Technician": "Other IT",
    "It Support Specialist": "Other IT",
    "It Technician": "Other IT",
    "Project Coordinator": "Other IT",
    "It Director":"Other IT",
    
    "Scrum Master": "Project Manager",
    "Technical Project Manager": "Project Manager",

    # --- Other No IT (8) ---
    "Civil Engineer": "Other No IT",
    "Electrical Engineering": "Other No IT",
    "Mechanical Engineer": "Other No IT",
    "Administrative Assistant": "Other No IT",
    "Arts": "Other No IT",
    "Customer Service Representative": "Other No IT",
    "Health And Fitness": "Other No IT",
    "Sales": "Other No IT",
    "Security Officer": "Other No IT",
    "Hr": "Other No IT",
    "Sales Associate": "Other No IT",
    "Advocate": "Other No IT",
    # --- Big Data Cloud Engineer ---
    "Devops Engineer": "Big Data Cloud Developer",
    "Cloud Engineer":"Big Data Cloud Developer",
    "Hadoop": "Big Data Cloud Developer",
    "Hadoop Developer": "Big Data Cloud Developer",
    "Sr. Hadoop Developer": "Big Data Cloud Developer",
    "Etl Developer": "Big Data Cloud Developer",

    # --- Data Analyst ---
    "Data Analyst ": "Data Analyst ",
    "It Business Analyst": "Data Analyst",
    "Data Science":"Data Scientist",
     # --- Consultant---
    "Consultant Consultant":"Consultant",
    "Consultant Consultant Consultant":"Consultant",
    "Contractor Contractor Contractor":"Consultant",
    "Independent Contractor":"Consultant",

    "Mobile App Developer (Ios/Android)": "Mobile Developer",
    "Android Application Developer":"Mobile Developer",
    "Android Developer":"Mobile Developer",
    
    "Business Analyst": "Data Analyst",
    "It Analyst": "Data Analyst",
    "Salesforce Administrator'": "Salesforce Developer",
    "Salesforce Admin/ Developer": "Salesforce Developer",
    "Salesforce Lightning Developer": "Salesforce Developer",
    "Frontend Developer": "Front End Developer",
    # "Sr. Front End Developer": "Senior Front End Developer", 
    # "Sr. Ui Developer": "Senior Front End Developer",
    "Ui Developer Ui Developer Ui Developer": "Ui Developer",  
    "Front- End Developer":"Front End Developer",
    "Front End Engineer":"Front End Developer",
    "Front End/Angular Developer":"Front End Developer",
    "Front End/Ui Developer":"Front End Ui Developer",
    "Front- End Web Developer":"Front End Developer",
    "Front-End Web Developer":"Front End Developer",
    "Developer Developer": "Software Developer",  
    "Lead Front End Developer":"Lead Developer",
    "Lead Java Developer": "Lead Developer",
    "Freelance Web Developer": "Freelance Developer",
    "Freelance Front End Developer":"Freelance Developer", 
    #                    
    "It Security Engineer": "Security Engineer",                               
    
    "It Project Coordinator": "Project Manager",
    "Senior It Project Manager": "Senior Project Manager",
    "It Consultant": "Consultant",
    "It Specialist": "Other IT" ,
}
# Remplacement 
c_resume_dataset=resume_dataset.copy()
c_resume_dataset['Category'] = c_resume_dataset['Category'].replace(map_remplacement)
# Suppression des lignes où Category == "Web Developer,Software Developer"
c_resume_dataset = c_resume_dataset[c_resume_dataset['Category'] != "Web Developer,Software Developer"]

# Avant traitement
print("Avant groupement:")
print(resume_dataset['Category'].nunique(), "catégories uniques")
# Après traitement
print("\nAprès normalisation:")
print(c_resume_dataset['Category'].nunique(), "catégories uniques")
# # Vérification
# print("Nombre de catégories par groupe après remplacement :")
# print(c_resume_dataset['Category'].value_counts())


Avant groupement:
182 catégories uniques

Après normalisation:
122 catégories uniques


In [34]:
# Identifier les catégories à supprimer (<30 occurrences)
new_rare_categories = category_counts[(category_counts < 30) ].index
# Filtrer le DataFrame pour ne garder que les lignes avec des catégories valides
filtered_dataset = c_resume_dataset[~c_resume_dataset['Category'].isin(new_rare_categories)]

# Vérification
print(f"Taille originale : {len(c_resume_dataset)}")
print(f"\nNombre de catégories : {len(c_resume_dataset['Category'].value_counts())}")
print(f"Taille après filtrage : {len(filtered_dataset)}")
print(f"\nNombre de catégories : {len(filtered_dataset['Category'].value_counts())}")

Taille originale : 11456

Nombre de catégories : 122
Taille après filtrage : 10440

Nombre de catégories : 48


In [35]:
print("\nCatégories conservées :")
print(filtered_dataset['Category'].value_counts())


Catégories conservées :
Category
Network Administrator                   1056
Security Analyst                         915
Software Developer                       853
Database Administrator                   660
Project Manager                          649
Python Developer,Software Developer      521
Systems Administrator                    491
Front End Developer                      461
Python Developer                         423
Software Developer,Web Developer         413
Java Developer,Software Developer        388
Java Developer                           358
Oracle Database Administrator            256
Project Manager,Software Developer       242
It Security Analyst                      230
It Project Manager                       204
Senior Java Developer                    174
Other IT                                 169
Senior Python Developer                  167
Job Seeker                               142
Network Engineer                         140
Full Stack Java Devel

#### Doublons

In [36]:
# Vérification des doublons
print(f"Nombre de doublons exacts: {filtered_dataset.duplicated().sum()}")

# Vérification des doublons potentiels (même texte mais catégories différentes)
duplicate_texts = filtered_dataset[filtered_dataset.duplicated(subset=['Resume'], keep=False)]
print(f"\nExemples de textes dupliqués avec catégories différentes:")
duplicate_texts.sort_values('Resume').head(3)

Nombre de doublons exacts: 0

Exemples de textes dupliqués avec catégories différentes:


,Category,Resume


#### Enregistrement dataset apres analyse et exploration(Normalisation)

In [ ]:
# filtered_dataset.to_csv(data_dir/'processed/cleaned_combined_resume_1.csv')

##### Preprocessing de base dataset

In [38]:
def preprocess_text(text):
    # Mise en minuscules
    text = text.lower()
    # Suppression :
    text = re.sub(r'https?://\S+|www\.\S+', '', text) #des URLs
    text = re.sub(r'<[^>]+>', '', text)# des balises HTML
    #text = re.sub(r'[\[\]\|@#$%^&*~]', '', text) 
    text = re.sub(r'[\[\]\|@#$%^&*~_+=<>/\\{}¦©®™]', '', text)#caractères non-linguistiques
    text = re.sub(r'--+', ' ', text)  #les suites de tirets
    text = re.sub(r'\d{6,}', ' ', text) #des longues séquences numériques (6 chiffres et plus)
    text = emoji.replace_emoji(text, replace='') # Suppression des emojis
    text = re.sub(r'\s+', ' ', text).strip()     # des espaces multiples
    return text

In [39]:
def clean_dataset(data)->pd.DataFrame:
    new_data=data.copy(deep=True)
    new_data.loc[:, 'Resume']=new_data['Resume'].apply(lambda x:preprocess_text(x))
    return new_data

In [40]:
cleaned_resume_dataset=clean_dataset(filtered_dataset)
cleaned_resume_dataset.head(5)

,Category,Resume
0,Database Administrator,"ability: installation and building server, run..."
1,Database Administrator,ability: database management systems administr...
2,Oracle Database Administrator,ability: over 4 years of experience as archite...
3,Oracle Database Administrator,"ability: oracle database administration, datab..."
4,Oracle Database Administrator,"ability: oracle database administration, oracl..."


In [41]:
filtered_dataset.head(5)

,Category,Resume
0,Database Administrator,"Ability: Installation and Building Server, Run..."
1,Database Administrator,Ability: database management systems administr...
2,Oracle Database Administrator,Ability: Over 4+ years of Experience as Archit...
3,Oracle Database Administrator,"Ability: Oracle Database Administration, Datab..."
4,Oracle Database Administrator,"Ability: Oracle Database Administration, Oracl..."


In [42]:

read_data(cleaned_resume_dataset)


 __Aperçu des données :
                        Category  \
0         Database Administrator   
1         Database Administrator   
2  Oracle Database Administrator   

                                              Resume  
0  ability: installation and building server, run...  
1  ability: database management systems administr...  
2  ability: over 4 years of experience as archite...  

 __Informations sur les données :
<class 'pandas.core.frame.DataFrame'>
Index: 10440 entries, 0 to 12103
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  10440 non-null  object
 1   Resume    10440 non-null  object
dtypes: object(2)
memory usage: 244.7+ KB
None
 
 __Taille : 
 (10440, 2)

 __Information sur les Categories: 
                                 Category  count
0                  Network Administrator   1056
1                       Security Analyst    915
2                     Software Developer    853
3               

In [43]:
#  catégories à supprimer
categories_to_remove = ['It Security Analyst', 'It Project Manager']

# Suppression effective
cleaned_resume_dataset = cleaned_resume_dataset[~cleaned_resume_dataset['Category'].isin(categories_to_remove)].copy(deep=True)
print("Nombre  CV : ",len(cleaned_resume_dataset['Resume'].value_counts()))
print("Nombre  Categories : ",len(cleaned_resume_dataset['Category'].value_counts()))
print("Liste Categories : ",cleaned_resume_dataset['Category'].value_counts().index.sort_values())

Nombre  CV :  10005
Nombre  Categories :  46
Liste Categories :  Index(['Big Data Cloud Developer', 'Consultant', 'Cyber Security Analyst',
       'Data Scientist', 'Database Administrator', 'Freelance Developer',
       'Front End Developer', 'Front End Web Developer', 'Front-End Developer',
       'Full Stack Developer', 'Full Stack Java Developer',
       'Information Security Analyst', 'It Manager', 'Java Developer',
       'Java Developer,Software Developer', 'Java Full Stack Developer',
       'Java/J2Ee Developer', 'Job Seeker', 'Mobile Developer',
       'Network Administrator', 'Network Engineer',
       'Oracle Database Administrator', 'Other IT', 'Other No IT',
       'Project Manager', 'Project Manager,Software Developer',
       'Python Developer', 'Python Developer,Software Developer',
       'Security Analyst', 'Senior Database Administrator',
       'Senior Front End Developer', 'Senior Java Developer',
       'Senior Java Full Stack Developer', 'Senior Java/J2Ee Develo

In [ ]:
#### Enregistrement Dataset

In [44]:
c_resume_dataset = c_resume_dataset[c_resume_dataset['Category'] != "Web Developer,Software Developer"]

In [45]:

cleaned_resume_dataset.to_csv(data_dir/'processed/cleaned_combined_resume_final.csv')
# read_data(resume_dataset)